In [ ]:

import pandas as pd
import numpy as np

# Load NAV History Data
nav_df = pd.read_csv(
    r"C:\Users\ELCOT\Documents\MutualFundProject\data\processed\nav_history_clean.csv"
)

# Convert date column
nav_df["date"] = pd.to_datetime(nav_df["date"])

# Sort records for return calculation
nav_df = nav_df.sort_values(
    by=["amfi_code", "date"]
)

# Calculate Daily Returns
nav_df["daily_return"] = (
    nav_df.groupby("amfi_code")["nav"]
          .pct_change()
)

# Function to Calculate VaR and CVaR
def calculate_var_cvar(returns):

    returns = returns.dropna()

    if len(returns) == 0:
        return pd.Series({
            "VaR_95": np.nan,
            "CVaR_95": np.nan
        })

    var_95 = np.percentile(returns, 5)

    cvar_95 = returns[
        returns <= var_95
    ].mean()

    return pd.Series({
        "VaR_95": round(var_95, 6),
        "CVaR_95": round(cvar_95, 6)
    })

# Generate VaR and CVaR Report

var_cvar_report = (
    nav_df.groupby("amfi_code")["daily_return"]
          .apply(calculate_var_cvar)
          .unstack()
          .reset_index()
)


# Save Output
var_cvar_report.to_csv(
    r"C:\Users\ELCOT\Documents\MutualFundProject\reports\var_cvar_report.csv",
    index=False
)


print("VaR & CVaR Report Generated Successfully")
print(var_cvar_report.head())

highest_risk_fund = (
    var_cvar_report
    .sort_values("VaR_95")
    .iloc[0]
)

print(
    f"Highest Risk Fund: {highest_risk_fund['amfi_code']} "
    f"with VaR of {highest_risk_fund['VaR_95']:.4f}"
)

In [ ]:


import pandas as pd
import numpy as np
from pathlib import Path


PROJECT_DIR = Path(
    r"C:\Users\ELCOT\Documents\MutualFundProject"
)

DATA_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports"

REPORT_DIR.mkdir(exist_ok=True)

# ----------------------------------------------------------
# Load NAV History Data
# ----------------------------------------------------------

nav_df = pd.read_csv(
    DATA_DIR / "nav_history_clean.csv"
)

# ----------------------------------------------------------
# Data Preparation
# ----------------------------------------------------------

nav_df["date"] = pd.to_datetime(
    nav_df["date"]
)

nav_df = nav_df.sort_values(
    by=["amfi_code", "date"]
)

# ----------------------------------------------------------
# Calculate Daily Returns
# ----------------------------------------------------------

nav_df["daily_return"] = (
    nav_df.groupby("amfi_code")["nav"]
          .pct_change()
)

# ----------------------------------------------------------
# Function to Compute VaR & CVaR
# ----------------------------------------------------------

def calculate_var_cvar(returns):

    returns = returns.dropna()

    if len(returns) == 0:
        return pd.Series({
            "VaR_95": np.nan,
            "CVaR_95": np.nan
        })

    # Historical VaR (95%)
    var_95 = np.percentile(
        returns,
        5
    )

    # Historical CVaR (95%)
    cvar_95 = returns[
        returns <= var_95
    ].mean()

    return pd.Series({
        "VaR_95": round(var_95, 6),
        "CVaR_95": round(cvar_95, 6)
    })

# ----------------------------------------------------------
# Generate Risk Metrics Report
# ----------------------------------------------------------

var_cvar_report = (
    nav_df.groupby("amfi_code")["daily_return"]
          .apply(calculate_var_cvar)
          .unstack()
          .reset_index()
)

# ----------------------------------------------------------
# Rank Funds by Risk
# ----------------------------------------------------------

var_cvar_report["VaR_Rank"] = (
    var_cvar_report["VaR_95"]
    .rank(method="dense")
)

var_cvar_report["CVaR_Rank"] = (
    var_cvar_report["CVaR_95"]
    .rank(method="dense")
)

# ----------------------------------------------------------
# Save Output
# ----------------------------------------------------------

var_cvar_report.to_csv(
    REPORT_DIR / "var_cvar_report.csv",
    index=False
)

# ----------------------------------------------------------
# Display Results
# ----------------------------------------------------------

print("=" * 60)
print("HISTORICAL VaR & CVaR REPORT")
print("=" * 60)

print(var_cvar_report.head())

print("\nReport Shape:")
print(var_cvar_report.shape)

# ----------------------------------------------------------
# Top 10 Highest Risk Funds
# ----------------------------------------------------------

top_risky_funds = (
    var_cvar_report
    .sort_values(
        by="VaR_95"
    )
    .head(10)
)

print("\nTop 10 Highest Risk Funds")
print(top_risky_funds)

# ----------------------------------------------------------
# Key Insight
# ----------------------------------------------------------

highest_risk_fund = (
    var_cvar_report
    .sort_values(
        by="VaR_95"
    )
    .iloc[0]
)

print("\nKey Insight")
print(
    f"Fund {highest_risk_fund['amfi_code']} "
    f"has the highest downside risk "
    f"with VaR of {highest_risk_fund['VaR_95']:.4f}"
)

print("\nReport saved successfully:")
print(REPORT_DIR / "var_cvar_report.csv")

In [ ]:
# ==========================================================
# DAY 6 - ROLLING 90 DAY SHARPE RATIO ANALYSIS
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ----------------------------------------------------------
# Project Paths
# ----------------------------------------------------------

PROJECT_DIR = Path(
    r"C:\Users\ELCOT\Documents\MutualFundProject"
)

DATA_DIR = PROJECT_DIR / "data" / "processed"
CHART_DIR = PROJECT_DIR / "charts"

CHART_DIR.mkdir(exist_ok=True)

# ----------------------------------------------------------
# Load NAV Data
# ----------------------------------------------------------

nav_df = pd.read_csv(
    DATA_DIR / "nav_history_clean.csv"
)

nav_df["date"] = pd.to_datetime(
    nav_df["date"]
)

nav_df = nav_df.sort_values(
    ["amfi_code", "date"]
)

# ----------------------------------------------------------
# Select Top 3 Funds
# ----------------------------------------------------------

selected_funds = (
    nav_df["amfi_code"]
    .unique()[:3]
)

# ----------------------------------------------------------
# Plot
# ----------------------------------------------------------

plt.figure(figsize=(14, 7))

for fund_code in selected_funds:

    fund_data = (
        nav_df[
            nav_df["amfi_code"] == fund_code
        ]
        .copy()
    )

    # Daily Returns
    fund_data["daily_return"] = (
        fund_data["nav"]
        .pct_change()
    )

    # Rolling Sharpe Ratio
    fund_data["rolling_sharpe"] = (
        fund_data["daily_return"]
        .rolling(window=90)
        .mean()
        /
        fund_data["daily_return"]
        .rolling(window=90)
        .std()
    ) * np.sqrt(252)

    # Monthly Average Sharpe
    fund_data["month"] = (
        fund_data["date"]
        .dt.to_period("M")
    )

    monthly_sharpe = (
        fund_data.groupby("month")
                 ["rolling_sharpe"]
                 .mean()
                 .reset_index()
    )

    plt.plot(
        monthly_sharpe["month"].astype(str),
        monthly_sharpe["rolling_sharpe"],
        linewidth=2,
        label=f"Fund {fund_code}"
    )

# ----------------------------------------------------------
# Chart Formatting
# ----------------------------------------------------------

plt.title(
    "90-Day Rolling Sharpe Ratio (Monthly Average)",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Month")
plt.ylabel("Sharpe Ratio")

plt.grid(alpha=0.3)

plt.legend(
    title="Funds",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.xticks(rotation=45)

plt.tight_layout()

# ----------------------------------------------------------
# Save Chart
# ----------------------------------------------------------

plt.savefig(
    CHART_DIR / "rolling_sharpe_chart.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Rolling Sharpe Chart Saved Successfully")

In [ ]:
latest_sharpe = []

for fund_code in selected_funds:

    fund_data = nav_df[
        nav_df["amfi_code"] == fund_code
    ].copy()

    fund_data["daily_return"] = (
        fund_data["nav"]
        .pct_change()
    )

    fund_data["rolling_sharpe"] = (
        fund_data["daily_return"]
        .rolling(90)
        .mean()
        /
        fund_data["daily_return"]
        .rolling(90)
        .std()
    ) * np.sqrt(252)

    latest_sharpe.append({
        "amfi_code": fund_code,
        "latest_sharpe":
        fund_data["rolling_sharpe"].iloc[-1]
    })

sharpe_summary = pd.DataFrame(
    latest_sharpe
)

print(sharpe_summary)

In [ ]:

import pandas as pd
from pathlib import Path

# Project Paths
PROJECT_DIR = Path(
    r"C:\Users\ELCOT\Documents\MutualFundProject"
)

DATA_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports"

REPORT_DIR.mkdir(exist_ok=True)

# Load Dataset
transactions_df = pd.read_csv(
    DATA_DIR / "investor_transactions_clean.csv"
)

transactions_df["transaction_date"] = pd.to_datetime(
    transactions_df["transaction_date"]
)

print(transactions_df.shape)
transactions_df.head()

In [ ]:
first_transaction = (
    transactions_df
    .groupby("investor_id")["transaction_date"]
    .min()
    .reset_index()
)

first_transaction["cohort_year"] = (
    first_transaction["transaction_date"]
    .dt.year
)

first_transaction.head()

In [ ]:
transactions_df = transactions_df.merge(
    first_transaction[
        ["investor_id", "cohort_year"]
    ],
    on="investor_id",
    how="left"
)

transactions_df.head()

In [ ]:
sip_transactions = (
    transactions_df[
        transactions_df["transaction_type"]
        .str.upper()
        == "SIP"
    ]
)

In [ ]:
sip_transactions = (
    transactions_df[
        transactions_df["transaction_type"]
        .str.lower()
        == "sip"
    ]
)

In [ ]:
cohort_summary = (
    sip_transactions
    .groupby("cohort_year")
    .agg(
        average_sip_amount=(
            "amount_inr",
            "mean"
        ),

        total_invested=(
            "amount_inr",
            "sum"
        ),

        investor_count=(
            "investor_id",
            "nunique"
        )
    )
    .reset_index()
)

cohort_summary

In [ ]:
fund_preference = pd.pivot_table(
    sip_transactions,
    index="cohort_year",
    columns="amfi_code",
    values="amount_inr",
    aggfunc="sum",
    fill_value=0
)

fund_preference.head()

In [ ]:
for cohort in fund_preference.index:
    
    print(f"\nTop Funds for Cohort {cohort}")

    print(
        fund_preference.loc[cohort]
        .sort_values(ascending=False)
        .head(5)
    )
    top_funds_2024 = (
    fund_preference.loc[2024]
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)

top_funds_2024.columns = [
    "amfi_code",
    "total_investment"
]

top_funds_2024 = top_funds_2024.merge(
    fund_master,
    on="amfi_code",
    how="left"
)

top_funds_2024


In [ ]:
cohort_summary.to_csv(
    REPORT_DIR / "cohort_analysis.csv",
    index=False
)

print("Cohort Analysis Saved Successfully")

In [ ]:
# ==========================================================
# TASK 4 - SIP CONTINUITY ANALYSIS
# ==========================================================
# Objective:
# Identify investors likely to discontinue SIPs
#
# Rules:
# - Consider only SIP transactions
# - Minimum 6 SIP transactions
# - Compute average gap between SIPs
# - Gap > 35 days => At-Risk
# ==========================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ----------------------------------------------------------
# Project Paths
# ----------------------------------------------------------

PROJECT_DIR = Path(
    r"C:\Users\ELCOT\Documents\MutualFundProject"
)

DATA_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports"

REPORT_DIR.mkdir(exist_ok=True)

# ----------------------------------------------------------
# Load Data
# ----------------------------------------------------------

transactions_df = pd.read_csv(
    DATA_DIR / "investor_transactions_clean.csv"
)

transactions_df["transaction_date"] = pd.to_datetime(
    transactions_df["transaction_date"]
)

# ----------------------------------------------------------
# Filter SIP Transactions
# ----------------------------------------------------------

sip_df = (
    transactions_df[
        transactions_df["transaction_type"]
        .str.lower()
        == "sip"
    ]
    .copy()
)

# ----------------------------------------------------------
# Sort Transactions
# ----------------------------------------------------------

sip_df = sip_df.sort_values(
    by=["investor_id", "transaction_date"]
)

# ----------------------------------------------------------
# Calculate Gap Between Transactions
# ----------------------------------------------------------

sip_df["gap_days"] = (
    sip_df.groupby("investor_id")
          ["transaction_date"]
          .diff()
          .dt.days
)

# ----------------------------------------------------------
# Investor SIP Statistics
# ----------------------------------------------------------

sip_continuity = (
    sip_df.groupby("investor_id")
          .agg(
              sip_transaction_count=(
                  "transaction_date",
                  "count"
              ),

              average_gap_days=(
                  "gap_days",
                  "mean"
              ),

              total_sip_amount=(
                  "amount_inr",
                  "sum"
              )
          )
          .reset_index()
)

# ----------------------------------------------------------
# Keep Investors with 6+ SIP Transactions
# ----------------------------------------------------------

sip_continuity = (
    sip_continuity[
        sip_continuity["sip_transaction_count"] >= 6
    ]
)

# ----------------------------------------------------------
# Risk Flag
# ----------------------------------------------------------

sip_continuity["status"] = np.where(
    sip_continuity["average_gap_days"] > 35,
    "At-Risk",
    "Healthy"
)

# ----------------------------------------------------------
# Round Values
# ----------------------------------------------------------

sip_continuity["average_gap_days"] = (
    sip_continuity["average_gap_days"]
    .round(2)
)

# ----------------------------------------------------------
# Sort by Gap
# ----------------------------------------------------------

sip_continuity = (
    sip_continuity.sort_values(
        by="average_gap_days",
        ascending=False
    )
)

# ----------------------------------------------------------
# Save Output
# ----------------------------------------------------------

sip_continuity.to_csv(
    REPORT_DIR / "sip_continuity.csv",
    index=False
)

# ----------------------------------------------------------
# Summary Statistics
# ----------------------------------------------------------

total_investors = len(sip_continuity)

at_risk_count = (
    sip_continuity["status"]
    .eq("At-Risk")
    .sum()
)

healthy_count = (
    sip_continuity["status"]
    .eq("Healthy")
    .sum()
)
risk_percentage = round(
    (at_risk_count / total_investors) * 100,
    2
)

print(
    f"{risk_percentage}% of investors are classified as At-Risk based on SIP continuity."
)
print("=" * 60)
print("SIP CONTINUITY ANALYSIS")
print("=" * 60)

print(f"Total Investors Analysed : {total_investors}")
print(f"At-Risk Investors        : {at_risk_count}")
print(f"Healthy Investors        : {healthy_count}")

print("\nTop 10 Highest Gap Investors")
print(
    sip_continuity.head(10)
)

print("\nFile Saved Successfully")
print("reports/sip_continuity.csv")

In [ ]:
# ==========================================================
# TASK 6 - SECTOR CONCENTRATION ANALYSIS (HHI)
# ==========================================================
# Objective:
# Compute Herfindahl-Hirschman Index (HHI)
# HHI = Sum of (Sector Weight)^2
#
# High HHI  -> Concentrated Portfolio
# Low HHI   -> Diversified Portfolio
# ==========================================================

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ----------------------------------------------------------
# Project Paths
# ----------------------------------------------------------

PROJECT_DIR = Path(
    r"C:\Users\ELCOT\Documents\MutualFundProject"
)

DATA_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports"
CHART_DIR = PROJECT_DIR / "charts"

REPORT_DIR.mkdir(exist_ok=True)
CHART_DIR.mkdir(exist_ok=True)

# ----------------------------------------------------------
# Load Portfolio Holdings Data
# ----------------------------------------------------------

holdings_df = pd.read_csv(
    DATA_DIR / "portfolio_holdings_clean.csv"
)

print("Dataset Shape:", holdings_df.shape)

# ----------------------------------------------------------
# Aggregate Sector Weights by Fund
# ----------------------------------------------------------

sector_weights = (
    holdings_df
    .groupby(
        ["amfi_code", "sector"]
    )["weight_pct"]
    .sum()
    .reset_index()
)

print("Sector Aggregation Completed")

# ----------------------------------------------------------
# Compute HHI
# ----------------------------------------------------------

sector_weights["weight_squared"] = (
    sector_weights["weight_pct"] ** 2
)

sector_hhi = (
    sector_weights
    .groupby("amfi_code")["weight_squared"]
    .sum()
    .reset_index()
)

sector_hhi.columns = [
    "amfi_code",
    "HHI"
]

# ----------------------------------------------------------
# Risk Classification
# ----------------------------------------------------------

sector_hhi["concentration_level"] = pd.cut(
    sector_hhi["HHI"],
    bins=[-1, 1500, 2500, float("inf")],
    labels=[
        "Diversified",
        "Moderately Concentrated",
        "Highly Concentrated"
    ]
)

# ----------------------------------------------------------
# Sort by HHI
# ----------------------------------------------------------

sector_hhi = (
    sector_hhi
    .sort_values(
        by="HHI",
        ascending=False
    )
)

# ----------------------------------------------------------
# Save CSV
# ----------------------------------------------------------

sector_hhi.to_csv(
    REPORT_DIR / "sector_hhi.csv",
    index=False
)

print("sector_hhi.csv Saved Successfully")

# ----------------------------------------------------------
# Top 10 Most Concentrated Funds
# ----------------------------------------------------------

top10 = sector_hhi.head(10)

# ----------------------------------------------------------
# Plot Chart
# ----------------------------------------------------------

plt.figure(figsize=(12, 6))

plt.bar(
    top10["amfi_code"].astype(str),
    top10["HHI"]
)

plt.title(
    "Top 10 Funds by Sector Concentration (HHI)"
)

plt.xlabel("AMFI Code")
plt.ylabel("HHI")

plt.xticks(rotation=45)

plt.tight_layout()

plt.savefig(
    CHART_DIR / "sector_hhi_chart.png",
    dpi=300
)

plt.show()

print("sector_hhi_chart.png Saved Successfully")

# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------

print("\nTop 10 Most Concentrated Funds")
print(top10)

print("\nTask 6 Completed Successfully")

Advanced Analytics Summary 

Key Insight 1: Highest Downside Risk Fund (VaR & CVaR)

The VaR and CVaR analysis identified Fund 119599 as the highest risk fund in the portfolio. It recorded a VaR (95%) of -0.0269 and CVaR (95%) of -0.0324, indicating higher potential losses during extreme market conditions compared to other funds.

Key Insight 2: Best Risk-Adjusted Performance (Sharpe Ratio)

Rolling Sharpe Ratio analysis shows that Fund 100033 achieved the highest performance with a Sharpe Ratio of 2.0575, indicating superior returns per unit of risk. In contrast, funds like 100016 (-1.2157) showed weaker risk-adjusted performance.

Key Insight 3: Investor Cohort Investment Behaviour

Cohort analysis reveals that the 2024 investor cohort significantly outperformed the 2025 cohort in total investments. The top fund for 2024 was AMFI 125498 (₹62,68,480), followed by 149322 and 119551, indicating stronger capital participation from earlier investors.

Key Insight 4: SIP Continuity and Investor Retention Risk

The SIP continuity analysis shows a major retention concern:

Total investors analysed: 1,362
At-Risk investors: 1,332 (97.8%)
Healthy investors: 30
Maximum SIP gap observed: 102.6 days

This indicates a high probability of SIP discontinuation and highlights the need for stronger investor engagement strategies.

Key Insight 5: Sector Concentration Risk (HHI Analysis)

Sector concentration analysis using HHI revealed significant differences in portfolio diversification.
The most concentrated fund was AMFI 119092 with HHI = 2967.69, followed by 148569 (2549.92) and 125498 (2531.55), indicating strong dependency on limited sectors. Funds like 100033 (2276.47) showed comparatively better diversification.

Final Conclusion

The advanced analytics framework successfully integrates:

Market Risk Analysis (VaR & CVaR)
Performance Evaluation (Sharpe Ratio)
Investor Behaviour Analysis (Cohort Analysis)
SIP Continuity & Retention Risk
Portfolio Concentration Risk (HHI)

These insights provide a complete view of fund performance, investor behaviour, and risk exposure, enabling better investment decisions and portfolio optimization.